# Company Co-Mention Analysis

This notebook analyzes a network of companies co-mentioned in news articles.  
Nodes are companies, edges are co-mentions, and edge weights count how often two companies appear together.

In [1]:
import pandas as pd
import networkx as nx

from networkx.algorithms.community import louvain_communities
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

In [2]:
co_mentions_df = pd.read_csv("company_comention_edges.csv")
companies_df = pd.read_csv("../../nyse_nasdaq_companies_all.csv")

print("Co-mentions rows:", len(co_mentions_df))
print("Companies rows:", len(companies_df))

print(co_mentions_df.head())
print(companies_df.head())

Co-mentions rows: 9703
Companies rows: 4184
      source target  weight    source_name target_name
0       Q312    Q95     126     Apple Inc.      Google
1       Q312  Q3884     119     Apple Inc.      Amazon
2      Q3884    Q95     108         Amazon      Google
3  Q20800404    Q95     108  Alphabet Inc.      Google
4      Q2283   Q312      88      Microsoft  Apple Inc.
   company_id                     company                 exchange ticker  \
0    Q1001788                Buenaventura  New York Stock Exchange    BVN   
1    Q1002992       Build-A-Bear Workshop  New York Stock Exchange    BBW   
2  Q100321332  United Nuclear Corporation  New York Stock Exchange    UNC   
3  Q100323973             Whitestone REIT  New York Stock Exchange    WSR   
4    Q1007000                     Genpact  New York Stock Exchange      G   

               industry  market_cap market_cap_date  \
0                mining         NaN             NaN   
1                retail         NaN             NaN  

In [3]:
G = nx.Graph()

for _, row in co_mentions_df.iterrows():
    G.add_edge(
        row["source"],
        row["target"],
        weight=row["weight"],
        source_name=row["source_name"],
        target_name=row["target_name"]
    )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1997
Edges: 9703


In [4]:
for node in G.nodes():
    G.nodes[node]["label"] = node

weighted_degree = dict(G.degree(weight="weight"))
nx.set_node_attributes(G, weighted_degree, "weighted_degree")

## Map industries into broader groups

The original industry labels are very detailed, so I group them into broader categories such as Finance, Energy, Technology, Healthcare, and Retail.

In [5]:
industry_group_terms = {
    "Finance": [
        "financial services",
        "financial service activities, except insurance and pension funding",
        "finance",
        "economics of banking",
        "bank",
        "investment",
        "asset management",
        "insurance",
        "insurance industry",
        "life insurance",
        "vehicle insurance",
        "health insurance",
        "health insurance company",
        "risk management",
        "wire transfer",
        "bitcoin",
        "international standard industrial classification",
        "financial sector",
        "fintech",
        "pension",
        "payment",
        "credit card",
        "capital market",
        "private equity",
        "venture capital",
    ],

    "Energy": [
        "petroleum industry",
        "energy industry",
        "energy company",
        "industrial gas",
        "photovoltaics",
        "photovoltaic system",
        "solar industry",
        "energy supply",
        "energy service company",
        "petroleum",
        "natural gas",
        "coal industry",
        "oilfield service",
        "list of oilfield service companies",
    ],

    "Utilities": [
        "public utility",
        "electricity supply company",
        "electricity generation",
        "electric power industry",
        "water supply",
        "water collection, treatment and supply",

    ],

    "Technology / Telecom": [
        "software industry",
        "software development",
        "enterprise software",
        "information technology",
        "information technology industry",
        "information technology consulting",
        "information and communications technology",
        "computer security",
        "information security",
        "computer and network surveillance",
        "computer industry",
        "computer hardware industry",
        "computer network",
        "computer storage media",
        "computer-aided design",
        "networking hardware",
        "internet",
        "web hosting service",
        "technology",
        "technology company",
        "artificial intelligence",
        "analytics",
        "automation",
        "robotics",
        "3d printing",
        "semiconductor industry",
        "electronics",
        "consumer electronics industry",
        "telecommunications",
        "communication",
        "video conference",
        "telepresence",
        "digital distribution",
        "electrical industry",
        "it service management",
        "cloud computing",
        "cloud storage",
        "software as a service",
        "business software industry",
        "internet industry",
        "electronics industry",
        "semiconductor",
        "computing",
        "online service",
        "peripheral",
        "electronic component",
        "voice over ip",
        "it infrastructure",
        "data center",
        "data analysis",
        "business intelligence",
        "managed security service",
        "high-performance computing",
        "wireless lan",
        "mobile telephony",
        "telephone company",
        "social networking service",
        "online video platform",
        "educational software",
        
        
    ],

    "Healthcare": [
        "pharmaceutical industry",
        "biotechnology",
        "biotechnology industry",
        "health care",
        "health technology",
        "medical technology industry",
        "medical equipment",
        "managed care",
        "life sciences",
        "high-performance liquid chromatography",
        "health care industry",
        "biopharmaceutical",
        "medical device",
        "medical device design",
        "pharmacy benefits manager",
        "senior living",
        "veterinary medicine",
        "health club",
    ],

    "Automotive": [
        "automotive industry",
        "car rental company",
        "automotive supplier",
        "car dealership",
        "autonomous car",
        "automotive part",
        "motor vehicle body manufacturing",
    ],

    "Industrials / Aerospace / Defense": [
        "industrial manufacturing",
        "industrial sector",
        "mechanical engineering",
        "engineering",
        "aerospace industry",
        "aerospace engineering",
        "aviation",
        "weapons industry",
        "defense contractor",
        "manufacture of machinery and equipment",
        "equipment rental",
        "power tool",
        "shipbuilding",
        "construction",
        "facility management",
        "outsourcing",
        "space industry",
        "heating, ventilation, and air conditioning",
        "bus manufacturing",
        "gas turbine",
        "maintenance",
    ],

    "Materials / Mining / Chemicals": [
        "chemical industry",
        "pesticide and other agricultural chemical manufacturing",
        "iron and steel industry",
        "mining",
        "mining industry",
        "metal",
        "cement industry",
        "building materials trade",
        "glass",
        "pulp and paper industry",
        "product packaging industry",
        "building material",
        "aluminium industry",
        "steel",
        "lithium",
        "zinc industry",
        "battery industry",
        "mineral",
    ],

    "Consumer / Retail / Food": [
        "retail",
        "wholesale",
        "trade",
        "e-commerce",
        "direct selling",
        "auction",
        "product distribution",
        "final good",
        "hardware store",
        "fast-moving consumer goods",
        "personal care product",
        "cosmetics industry",
        "food industry",
        "food processing",
        "food service",
        "restaurant",
        "fast food",
        "fast casual restaurant",
        "system catering",
        "beverage industry",
        "brewing industry",
        "coffee industry",
        "manufacture of cocoa, chocolate and sugar confectionery",
        "alcohol industry",
        "tobacco industry",
        "clothing industry",
        "footwear industry",
        "textile industry",
        "cannabis industry",
        "gastronomy",
        "fashion",
        "mattress",
        "bedding",
        "furniture",
        "discount store",
        "online shopping",
        "convenience store",
        "grocery store",
        "supermarket",
        "dairy",
        "bookselling",
        "wine",
        "outlet store",
        "sporting goods industry",
        "home appliance industry",
    ],

    "Media / Entertainment": [
        "media industry",
        "mass media",
        "show business",
        "streaming media",
        "broadcasting",
        "broadcast television system",
        "television",
        "terrestrial television",
        "radio broadcasting",
        "journalism",
        "music industry",
        "production music",
        "animation",
        "video game industry",
        "game industry",
        "sports industry",
        "gambling",
        "gambling industry",
         "entertainment",
        "entertainment industry",
        "publishing",
        "recording medium",
        "radio",
        "photography",
        "digital media",
        "news media",
        "cinematography",
        "professional sport",
        "video on demand",
        "leisure park industry",
    ],

    "Transport / Logistics": [
        "logistics",
        "transport",
        "transport industry",
        "freight transport industry",
        "air transport",
        "rail transport",
        "water transport",
        "shipping line",
        "waste management",
        "waste management industry",
        "freight transport",
        "trucking industry",
        "intermodal container",
        "supply chain management",
        "truck stop",
    ],

    "Real Estate": [
        "real estate industry",
        "real estate investment trust",
        "self storage",
        "real estate development",
        "real property",
        "property management",
        "timeshare",
        "apartment",
    ],

    "Travel / Hospitality": [
        "tourism",
        "tourism industry",
        "hospitality industry",
        "space tourism",
        "hotel industry",
        "travel agency",
    ],

    "Agriculture": [
        "agriculture",
        "agribusiness",
    ],

    "Professional Services": [
        "professional service",
        "consulting company",
        "marketing",
        "e-recruitment",
        "advertising",
        "online advertising",
        "corporate services",
        "business and other management consultancy activities",
        "market research",
        "employment services industry",
        "recruitment",
        "staffing industry",
        "funeral services industry",
        "pre-press and pre-media services",
    ],

    "Holding / Conglomerate": [
        "holding company",
        "holding company activities",
        "conglomerate",
    ],

    "Other / Unclear": [
        "tertiary sector of the economy",
        "quaternary sector of the economy",
        "education",
        "educational system",
        "for-profit education",
        "business-to-business",
        "service",
        "pest control",
    ],
}

In [7]:
industry_group_terms_lower = {
    group: {term.lower().strip() for term in terms}
    for group, terms in industry_group_terms.items()
}

def map_to_industry_group(industry):
    if pd.isna(industry):
        return "Unknown"

    industry = str(industry).lower().strip()

    for group, terms in industry_group_terms_lower.items():
        if industry in terms:
            return group

    return "Other / Unmapped"

companies_df["industry_group"] = companies_df["industry"].apply(map_to_industry_group)

companies_df[["company", "industry", "industry_group"]].head(20)

,company,industry,industry_group
0,Buenaventura,mining,Materials / Mining / Chemicals
1,Build-A-Bear Workshop,retail,Consumer / Retail / Food
2,United Nuclear Corporation,mining,Materials / Mining / Chemicals
3,Whitestone REIT,NaN,Unknown
4,Genpact,professional service,Professional Services
5,Bunge Limited,food industry,Consumer / Retail / Food
6,Pool Corporation,NaN,Unknown
7,Vontier,NaN,Unknown
8,Adecoagro,agribusiness,Agriculture
9,Aurora Innovation,autonomous car,Automotive


In [8]:
companies_df["industry_group"].value_counts()

industry_group
Unknown                              1218
Technology / Telecom                  527
Finance                               403
Consumer / Retail / Food              371
Healthcare                            342
Other / Unmapped                      279
Industrials / Aerospace / Defense     180
Energy                                179
Media / Entertainment                 133
Materials / Mining / Chemicals        131
Real Estate                            91
Automotive                             76
Transport / Logistics                  69
Utilities                              51
Professional Services                  38
Travel / Hospitality                   36
Holding / Conglomerate                 26
Other / Unclear                        23
Agriculture                            11
Name: count, dtype: int64

##  Add company metadata to graph nodes

The graph nodes are company IDs, so I use `company_id` to add company names, industries, and industry groups to each node.

In [9]:
# Make sure IDs are clean strings
companies_df["company_id"] = companies_df["company_id"].astype(str).str.strip()

# Your graph nodes are company IDs, so map by company_id
company_name_map = dict(
    zip(companies_df["company_id"], companies_df["company"])
)

industry_map = dict(
    zip(companies_df["company_id"], companies_df["industry"])
)

industry_group_map = dict(
    zip(companies_df["company_id"], companies_df["industry_group"])
)
revenue_map = dict(zip(companies_df["company_id"], companies_df["revenue"]))

market_cap_map = dict(zip(companies_df["company_id"], companies_df["market_cap"]))

for node in G.nodes():
    node_id = str(node).strip()

    G.nodes[node]["label"] = company_name_map.get(node_id, node_id)
    G.nodes[node]["industry"] = industry_map.get(node_id, "Unknown")
    G.nodes[node]["industry_group"] = industry_group_map.get(node_id, "Unknown")

In [10]:
node_industry_check = pd.DataFrame([
    {
        "company_id": node,
        "company": G.nodes[node].get("label", node),
        "industry": G.nodes[node].get("industry", "Unknown"),
        "industry_group": G.nodes[node].get("industry_group", "Unknown"),
    }
    for node in G.nodes()
])

node_industry_check.head(20)

,company_id,company,industry,industry_group
0,Q312,Apple Inc.,digital distribution,Technology / Telecom
1,Q95,Google,information technology,Technology / Telecom
2,Q3884,Amazon,retail,Consumer / Retail / Food
3,Q20800404,Alphabet Inc.,information technology,Technology / Telecom
4,Q2283,Microsoft,software development,Technology / Telecom
5,Q193326,Goldman Sachs,financial services,Finance
6,Q334204,Morgan Stanley,financial services,Finance
7,Q219508,Citigroup,bank,Finance
8,Q483551,Walmart,retail,Consumer / Retail / Food
9,Q372657,Credit Suisse,economics of banking,Finance


##  Louvain community detection

I use Louvain to find communities of companies that are strongly connected through repeated co-mentions.

In [11]:
communities = louvain_communities(
    G,
    weight="weight",
    resolution=1,
    seed=42
)

for cluster_id, community in enumerate(communities):
    for company in community:
        G.nodes[company]["cluster"] = cluster_id

print("Found clusters:", len(communities))

Found clusters: 46


## Create node table

I convert graph nodes into a dataframe containing company name, Louvain cluster, industry group, and weighted degree.

In [12]:
node_df = pd.DataFrame([
    {
        "company_id": node,
        "company": G.nodes[node].get("label", node),
        "cluster": G.nodes[node].get("cluster"),
        "industry": G.nodes[node].get("industry", "Unknown"),
        "industry_group": G.nodes[node].get("industry_group", "Unknown"),
        "weighted_degree": G.nodes[node].get("weighted_degree", 0),
        "revenue": revenue_map.get(str(node).strip(), pd.NA),
        "market_cap_2018": market_cap_map.get(str(node).strip(), pd.NA),
    }
    for node in G.nodes()
])

In [13]:
eval_df = node_df[
    ~node_df["industry_group"].isin(["Unknown", "Other / Unmapped"])
].copy()

print("Total graph nodes:", len(node_df))
print("Nodes used for evaluation:", len(eval_df))

eval_df.head()

Total graph nodes: 1997
Nodes used for evaluation: 1408


,company_id,company,cluster,industry,industry_group,weighted_degree,revenue,market_cap_2018
0,Q312,Apple Inc.,0,digital distribution,Technology / Telecom,1175,4.161610e+11,3.205000e+12
1,Q95,Google,0,information technology,Technology / Telecom,928,NaN,NaN
2,Q3884,Amazon,0,retail,Consumer / Retail / Food,1019,7.169240e+11,2.018000e+12
3,Q20800404,Alphabet Inc.,0,information technology,Technology / Telecom,614,4.028360e+11,1.961000e+12
4,Q2283,Microsoft,0,software development,Technology / Telecom,838,2.817240e+11,3.162000e+12


## Compare Louvain clusters with industry groups

I compare the detected graph communities with the known industry groups using a contingency table, purity, NMI, and ARI.

In [14]:
contingency = pd.crosstab(
    eval_df["cluster"],
    eval_df["industry_group"]
)

contingency

industry_group,Agriculture,Automotive,Consumer / Retail / Food,Energy,Finance,Healthcare,Holding / Conglomerate,Industrials / Aerospace / Defense,Materials / Mining / Chemicals,Media / Entertainment,Other / Unclear,Professional Services,Real Estate,Technology / Telecom,Transport / Logistics,Travel / Hospitality,Utilities
cluster,,,,,,,,,,,,,,,,,
0,0,3,21,6,16,6,0,5,3,23,1,3,5,95,5,3,2
1,0,0,2,0,1,0,0,2,0,0,0,0,0,0,0,0,0
2,1,2,7,7,70,4,0,7,9,3,1,2,0,14,2,1,1
3,0,12,19,14,39,40,3,23,6,14,6,2,13,75,7,4,12
4,0,21,7,2,8,0,0,5,0,1,0,0,1,11,3,0,2
5,0,0,4,0,0,0,0,0,0,3,0,0,1,3,0,1,0
6,2,1,37,6,13,8,0,7,16,1,2,1,0,15,4,5,7
7,0,1,6,2,1,71,0,4,4,0,0,0,4,6,1,0,2
8,1,1,1,49,7,0,0,5,7,2,0,0,2,6,7,0,3


In [15]:
purity = contingency.max(axis=1).sum() / contingency.values.sum()

print("Purity:", purity)

Purity: 0.4119318181818182


In [16]:
cluster_summary_rows = []

for cluster_id, group in eval_df.groupby("cluster"):
    industry_counts = group["industry_group"].value_counts()

    dominant_industry = industry_counts.index[0]
    dominant_count = industry_counts.iloc[0]
    cluster_size = len(group)

    cluster_summary_rows.append({
        "cluster": cluster_id,
        "cluster_size": cluster_size,
        "dominant_industry": dominant_industry,
        "dominant_count": dominant_count,
        "cluster_purity": dominant_count / cluster_size,
        "all_industries": dict(industry_counts),
    })

cluster_summary_df = pd.DataFrame(cluster_summary_rows)

cluster_summary_df = cluster_summary_df.sort_values(
    "cluster_size",
    ascending=False
)

cluster_summary_df

,cluster,cluster_size,dominant_industry,dominant_count,cluster_purity,all_industries
3,3,289,Technology / Telecom,75,0.259516,"{'Technology / Telecom': 75, 'Healthcare': 40,..."
0,0,197,Technology / Telecom,95,0.482234,"{'Technology / Telecom': 95, 'Media / Entertai..."
2,2,131,Finance,70,0.534351,"{'Finance': 70, 'Technology / Telecom': 14, 'M..."
6,6,125,Consumer / Retail / Food,37,0.296000,"{'Consumer / Retail / Food': 37, 'Materials / ..."
7,7,102,Healthcare,71,0.696078,"{'Healthcare': 71, 'Consumer / Retail / Food':..."
11,11,101,Industrials / Aerospace / Defense,22,0.217822,"{'Industrials / Aerospace / Defense': 22, 'Fin..."
9,9,99,Consumer / Retail / Food,54,0.545455,"{'Consumer / Retail / Food': 54, 'Media / Ente..."
8,8,91,Energy,49,0.538462,"{'Energy': 49, 'Materials / Mining / Chemicals..."
19,19,73,Finance,15,0.205479,"{'Finance': 15, 'Materials / Mining / Chemical..."
4,4,61,Automotive,21,0.344262,"{'Automotive': 21, 'Technology / Telecom': 11,..."


In [17]:
y_true_industry = eval_df["industry_group"]
y_pred_cluster = eval_df["cluster"]

nmi = normalized_mutual_info_score(y_true_industry, y_pred_cluster)
ari = adjusted_rand_score(y_true_industry, y_pred_cluster)

print("Purity:", purity)
print("NMI:", nmi)
print("ARI:", ari)

Purity: 0.4119318181818182
NMI: 0.20278140990226948
ARI: 0.10220316419838812


## Visualize the network

I visualize the same graph twice: once colored by Louvain cluster and once colored by industry group.

In [18]:
from ipysigma import Sigma

Sigma(
    G,
    node_label="label",          # company name
    node_color="cluster",        # Louvain cluster
    node_size="weighted_degree", # more connected companies are bigger
    edge_size="weight",          # stronger co-mentions are thicker
    default_edge_type="curve",
    clickable_edges=False
)

Sigma(nx.Graph with 1,997 nodes and 9,703 edges)

In [19]:
Sigma(
    G,
    node_label="label",
    node_color="industry_group", # industry categories
    node_size="weighted_degree",
    edge_size="weight",
    default_edge_type="curve",
    clickable_edges=False
)

Sigma(nx.Graph with 1,997 nodes and 9,703 edges)

## Robustness test with edge thresholds

I test whether removing weak co-mentions changes the agreement between Louvain clusters and industry groups.

In [20]:
results = []

for min_weight in [1, 2, 3, 5, 10]:
    filtered_edges = co_mentions_df[
        co_mentions_df["weight"] >= min_weight
    ].copy()

    G_temp = nx.Graph()

    for _, row in filtered_edges.iterrows():
        G_temp.add_edge(
            row["source"],
            row["target"],
            weight=row["weight"]
        )

    # skip empty graphs
    if G_temp.number_of_nodes() == 0 or G_temp.number_of_edges() == 0:
        continue

    # add node attributes
    for node in G_temp.nodes():
        node_id = str(node).strip()

        G_temp.nodes[node]["label"] = company_name_map.get(node_id, node_id)
        G_temp.nodes[node]["industry"] = industry_map.get(node_id, "Unknown")
        G_temp.nodes[node]["industry_group"] = industry_group_map.get(node_id, "Unknown")

    weighted_degree_temp = dict(G_temp.degree(weight="weight"))
    nx.set_node_attributes(G_temp, weighted_degree_temp, "weighted_degree")

    # Louvain
    communities_temp = louvain_communities(
        G_temp,
        weight="weight",
        resolution=1,
        seed=42
    )

    for cluster_id, community in enumerate(communities_temp):
        for node in community:
            G_temp.nodes[node]["cluster"] = cluster_id

    # node dataframe
    node_df_temp = pd.DataFrame([
        {
            "company_id": node,
            "company": G_temp.nodes[node].get("label", node),
            "cluster": G_temp.nodes[node].get("cluster"),
            "industry_group": G_temp.nodes[node].get("industry_group", "Unknown"),
            "weighted_degree": G_temp.nodes[node].get("weighted_degree", 0),
        }
        for node in G_temp.nodes()
    ])

    eval_temp = node_df_temp[
        ~node_df_temp["industry_group"].isin(["Unknown", "Other / Unmapped"])
    ].copy()

    if len(eval_temp) == 0:
        continue

    contingency_temp = pd.crosstab(
        eval_temp["cluster"],
        eval_temp["industry_group"]
    )

    purity_temp = (
        contingency_temp.max(axis=1).sum()
        / contingency_temp.values.sum()
    )

    nmi_temp = normalized_mutual_info_score(
        eval_temp["industry_group"],
        eval_temp["cluster"]
    )

    ari_temp = adjusted_rand_score(
        eval_temp["industry_group"],
        eval_temp["cluster"]
    )

    results.append({
        "min_edge_weight": min_weight,
        "nodes": G_temp.number_of_nodes(),
        "edges": G_temp.number_of_edges(),
        "evaluated_nodes": len(eval_temp),
        "clusters": len(communities_temp),
        "purity": purity_temp,
        "nmi": nmi_temp,
        "ari": ari_temp,
    })
   

threshold_results_df = pd.DataFrame(results)

threshold_results_df

,min_edge_weight,nodes,edges,evaluated_nodes,clusters,purity,nmi,ari
0,1,1997,9703,1408,46,0.411932,0.202781,0.102203
1,2,969,3030,720,54,0.415278,0.264249,0.095324
2,3,581,1525,457,50,0.459519,0.334426,0.116976
3,5,296,650,253,40,0.588933,0.476686,0.199292
4,10,135,224,124,26,0.701613,0.583921,0.272868


## Node centrality

I compute centrality measures to identify the most important companies in the co-mention network.

In [80]:
# Degree centrality: number of distinct neighbors, normalized
degree_centrality = nx.degree_centrality(G)

# Weighted degree: sum of edge weights
weighted_degree = dict(G.degree(weight="weight"))

for u, v, data in G.edges(data=True):
    data["distance"] = 1 / data["weight"]

betweenness = nx.betweenness_centrality(
    G,
    weight="distance",
    normalized=True
)
# PageRank: overall network importance
pagerank = nx.pagerank(
    G,
    weight="weight"
)

In [77]:
centrality_df = pd.DataFrame([
    {
        "company_id": node,
        "company": G.nodes[node].get("label", node),
        "industry_group": G.nodes[node].get("industry_group", "Unknown"),
        "degree": G.degree(node),
        "degree_centrality": degree_centrality[node],
        "weighted_degree": weighted_degree[node],
        "betweenness": betweenness[node],
        "pagerank": pagerank[node],
    }
    for node in G.nodes()
])

centrality_df.head()

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
0,Q312,Apple Inc.,Technology / Telecom,246,0.123246,1175,0.258159,0.019776
1,Q95,Google,Technology / Telecom,179,0.089679,928,0.146827,0.014813
2,Q3884,Amazon,Consumer / Retail / Food,207,0.103707,1019,0.256763,0.016822
3,Q20800404,Alphabet Inc.,Technology / Telecom,103,0.051603,614,0.082940,0.009445
4,Q2283,Microsoft,Technology / Telecom,196,0.098196,838,0.098145,0.014400


## Most central companies

I rank companies by weighted degree, betweenness, and PageRank.

In [78]:
centrality_df.sort_values(
    "weighted_degree",
    ascending=False
).head(20)

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
24,Q1472929,"Nasdaq, Inc.",Finance,633,0.317134,1508,0.537691,0.043951
0,Q312,Apple Inc.,Technology / Telecom,246,0.123246,1175,0.258159,0.019776
2,Q3884,Amazon,Consumer / Retail / Food,207,0.103707,1019,0.256763,0.016822
1,Q95,Google,Technology / Telecom,179,0.089679,928,0.146827,0.014813
4,Q2283,Microsoft,Technology / Telecom,196,0.098196,838,0.098145,0.014400
6,Q334204,Morgan Stanley,Finance,225,0.112725,817,0.179186,0.013932
5,Q193326,Goldman Sachs,Finance,199,0.099699,707,0.068028,0.011856
3,Q20800404,Alphabet Inc.,Technology / Telecom,103,0.051603,614,0.082940,0.009445
7,Q219508,Citigroup,Finance,148,0.074148,576,0.058280,0.009514
10,Q66048,Deutsche Bank,Finance,162,0.081162,570,0.043150,0.009589


In [79]:
centrality_df.sort_values(
    "betweenness",
    ascending=False
).head(20)

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
24,Q1472929,"Nasdaq, Inc.",Finance,633,0.317134,1508,0.537691,0.043951
0,Q312,Apple Inc.,Technology / Telecom,246,0.123246,1175,0.258159,0.019776
2,Q3884,Amazon,Consumer / Retail / Food,207,0.103707,1019,0.256763,0.016822
6,Q334204,Morgan Stanley,Finance,225,0.112725,817,0.179186,0.013932
1,Q95,Google,Technology / Telecom,179,0.089679,928,0.146827,0.014813
4,Q2283,Microsoft,Technology / Telecom,196,0.098196,838,0.098145,0.014400
3,Q20800404,Alphabet Inc.,Technology / Telecom,103,0.051603,614,0.082940,0.009445
26,Q66,Boeing,Industrials / Aerospace / Defense,133,0.066633,390,0.076945,0.007516
15,Q35476,AT&T,Technology / Telecom,140,0.070140,433,0.068925,0.007438
5,Q193326,Goldman Sachs,Finance,199,0.099699,707,0.068028,0.011856


In [81]:
centrality_df.sort_values(
    "pagerank",
    ascending=False
).head(20)

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank
24,Q1472929,"Nasdaq, Inc.",Finance,633,0.317134,1508,0.537691,0.043951
0,Q312,Apple Inc.,Technology / Telecom,246,0.123246,1175,0.258159,0.019776
2,Q3884,Amazon,Consumer / Retail / Food,207,0.103707,1019,0.256763,0.016822
1,Q95,Google,Technology / Telecom,179,0.089679,928,0.146827,0.014813
4,Q2283,Microsoft,Technology / Telecom,196,0.098196,838,0.098145,0.014400
6,Q334204,Morgan Stanley,Finance,225,0.112725,817,0.179186,0.013932
5,Q193326,Goldman Sachs,Finance,199,0.099699,707,0.068028,0.011856
10,Q66048,Deutsche Bank,Finance,162,0.081162,570,0.043150,0.009589
7,Q219508,Citigroup,Finance,148,0.074148,576,0.058280,0.009514
3,Q20800404,Alphabet Inc.,Technology / Telecom,103,0.051603,614,0.082940,0.009445


## Add revenue and market capitalization

I merge centrality results with company revenue and market capitalization to test whether larger companies are more central.

In [31]:
analysis_df = centrality_df.merge(
    companies_df[
        [
            "company_id",
            "company",
            "industry",
            "industry_group",
            "revenue",
            "market_cap"
        ]
    ],
    on="company_id",
    how="left",
    suffixes=("", "_meta")
)

analysis_df.head()

,company_id,company,industry_group,degree,degree_centrality,weighted_degree,betweenness,pagerank,company_meta,industry,industry_group_meta,revenue,market_cap
0,Q312,Apple Inc.,Technology / Telecom,246,0.123246,1175,0.258159,0.019776,Apple Inc.,digital distribution,Technology / Telecom,4.161610e+11,3.205000e+12
1,Q95,Google,Technology / Telecom,179,0.089679,928,0.146827,0.014813,Google,information technology,Technology / Telecom,NaN,NaN
2,Q3884,Amazon,Consumer / Retail / Food,207,0.103707,1019,0.256763,0.016822,Amazon,retail,Consumer / Retail / Food,7.169240e+11,2.018000e+12
3,Q20800404,Alphabet Inc.,Technology / Telecom,103,0.051603,614,0.082940,0.009445,Alphabet Inc.,information technology,Technology / Telecom,4.028360e+11,1.961000e+12
4,Q2283,Microsoft,Technology / Telecom,196,0.098196,838,0.098145,0.014400,Microsoft,software development,Technology / Telecom,2.817240e+11,3.162000e+12


In [32]:
analysis_df["revenue"] = pd.to_numeric(analysis_df["revenue"], errors="coerce")
analysis_df["market_cap"] = pd.to_numeric(analysis_df["market_cap"], errors="coerce")

analysis_df[["company", "revenue", "market_cap", "weighted_degree", "pagerank", "betweenness"]].head()

,company,revenue,market_cap,weighted_degree,pagerank,betweenness
0,Apple Inc.,4.161610e+11,3.205000e+12,1175,0.019776,0.258159
1,Google,NaN,NaN,928,0.014813,0.146827
2,Amazon,7.169240e+11,2.018000e+12,1019,0.016822,0.256763
3,Alphabet Inc.,4.028360e+11,1.961000e+12,614,0.009445,0.082940
4,Microsoft,2.817240e+11,3.162000e+12,838,0.014400,0.098145


## Company size and centrality

I use company-size quartiles and Kruskal-Wallis tests to evaluate whether larger companies tend to be more central. This is more appropriate than a simple linear correlation because centrality values are highly skewed and contain strong outliers.

In [82]:
import numpy as np

analysis_df["log_revenue"] = np.log1p(analysis_df["revenue"])
analysis_df["log_market_cap"] = np.log1p(analysis_df["market_cap"])
analysis_df["log_weighted_degree"] = np.log1p(analysis_df["weighted_degree"])
analysis_df["log_degree"] = np.log1p(analysis_df["degree"])
analysis_df["log_pagerank"] = np.log1p(analysis_df["pagerank"])
analysis_df["log_betweenness"] = np.log1p(analysis_df["betweenness"])

In [112]:
from scipy.stats import pearsonr
import numpy as np

# Use only companies with revenue
revenue_corr_df = analysis_df[
    analysis_df["revenue"].notna()
].copy()

# Optional but recommended because revenue and weighted degree are highly skewed
revenue_corr_df["log_revenue"] = np.log1p(revenue_corr_df["revenue"])
revenue_corr_df["log_weighted_degree"] = np.log1p(revenue_corr_df["weighted_degree"])

pearson_corr, pearson_p = pearsonr(
    revenue_corr_df["log_revenue"],
    revenue_corr_df["log_weighted_degree"]
)

print("Pearson correlation: log revenue vs log weighted degree")
print("Correlation:", pearson_corr)
print("p-value:", pearson_p)
print("N:", len(revenue_corr_df))

Pearson correlation: log revenue vs log weighted degree
Correlation: 0.3751182538220358
p-value: 1.0215851775814997e-16
N: 457


## Centrality by company size quartiles

I divide companies into revenue and market-cap quartiles and compare median centrality across groups.

In [84]:
revenue_df = analysis_df[analysis_df["revenue"].notna()].copy()

revenue_df["revenue_quartile"] = pd.qcut(
    revenue_df["revenue"],
    q=4,
    labels=["Q1 lowest revenue", "Q2", "Q3", "Q4 highest revenue"],
    duplicates="drop"
)

revenue_quartile_summary = revenue_df.groupby(
    "revenue_quartile",
    observed=True
).agg(
    n_companies=("company_id", "count"),
    median_degree=("degree", "median"),
    median_weighted_degree=("weighted_degree", "median"),
    median_pagerank=("pagerank", "median"),
    median_betweenness=("betweenness", "median")
).reset_index()

revenue_quartile_summary

,revenue_quartile,n_companies,median_degree,median_weighted_degree,median_pagerank,median_betweenness
0,Q1 lowest revenue,115,3.0,4.0,0.000199,0.000000e+00
1,Q2,114,6.0,8.0,0.000318,5.022576e-07
2,Q3,114,15.0,27.0,0.000607,2.586627e-05
3,Q4 highest revenue,114,22.0,48.5,0.001019,1.929925e-03


In [91]:
marketcap_df = analysis_df[analysis_df["market_cap"].notna()].copy()

marketcap_df["market_cap_quartile"] = pd.qcut(
    marketcap_df["market_cap"],
    q=4,
    labels=["Q1 lowest market cap", "Q2", "Q3", "Q4 highest market cap"],
    duplicates="drop"
)

marketcap_quartile_summary = marketcap_df.groupby(
    "market_cap_quartile",
    observed=True
).agg(
    n_companies=("company_id", "count"),
    median_degree=("degree", "median"),
    median_weighted_degree=("weighted_degree", "median"),
    median_pagerank=("pagerank", "median"),
    median_betweenness=("betweenness", "median")
).reset_index()

marketcap_quartile_summary

,market_cap_quartile,n_companies,median_degree,median_weighted_degree,median_pagerank,median_betweenness
0,Q1 lowest market cap,36,4.0,4.5,0.000200,0.000000
1,Q2,36,18.0,30.0,0.000733,0.000903
2,Q3,36,32.5,76.0,0.001546,0.004200
3,Q4 highest market cap,36,52.0,143.5,0.002578,0.007260


In [92]:
from scipy.stats import kruskal

groups = [
    group["weighted_degree"].dropna().values
    for _, group in revenue_df.groupby("revenue_quartile", observed=True)
    if len(group) > 0
]

stat, p = kruskal(*groups)

print("Revenue quartiles - weighted degree")
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Revenue quartiles - weighted degree
Kruskal-Wallis statistic: 124.86831361086547
p-value: 6.899118024100145e-27


In [93]:
groups = [
    group["weighted_degree"].dropna().values
    for _, group in marketcap_df.groupby("market_cap_quartile", observed=True)
    if len(group) > 0
]

stat, p = kruskal(*groups)

print("Market-cap quartiles - weighted degree")
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Market-cap quartiles - weighted degree
Kruskal-Wallis statistic: 59.66408371210041
p-value: 6.934411818771043e-13


## Centrality by industry

I compare centrality across industry groups using descriptive statistics and the Kruskal-Wallis test.

In [96]:
industry_analysis_df = analysis_df[
    ~analysis_df["industry_group"].isin(["Unknown", "Other / Unmapped"])
].copy()

groups = [
    group[metric].dropna().values
    for _, group in industry_analysis_df.groupby("industry_group")
    if len(group) >= 5
]

stat, p = kruskal(*groups)

print("Metric:", metric)
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Metric: weighted_degree
Kruskal-Wallis statistic: 36.63106284710053
p-value: 0.0023627355738155605


In [97]:
industry_centrality = industry_analysis_df.groupby("industry_group").agg(
    n_companies=("company_id", "count"),
    median_weighted_degree=("weighted_degree", "median"),
    mean_weighted_degree=("weighted_degree", "mean"),
    q1_weighted_degree=("weighted_degree", lambda x: x.quantile(0.25)),
    q3_weighted_degree=("weighted_degree", lambda x: x.quantile(0.75)),
    median_pagerank=("pagerank", "median"),
    median_betweenness=("betweenness", "median"),
).reset_index()

industry_centrality["iqr_weighted_degree"] = (
    industry_centrality["q3_weighted_degree"] 
    - industry_centrality["q1_weighted_degree"]
)

industry_centrality.sort_values("median_weighted_degree", ascending=False)

,industry_group,n_companies,median_weighted_degree,mean_weighted_degree,q1_weighted_degree,q3_weighted_degree,median_pagerank,median_betweenness,iqr_weighted_degree
1,Automotive,39,10.0,54.487179,3.00,44.50,0.000356,5.022576e-07,41.50
2,Consumer / Retail / Food,184,8.0,26.983696,3.00,23.00,0.000279,0.000000e+00,20.00
15,Travel / Hospitality,25,6.0,10.640000,3.00,15.00,0.000224,0.000000e+00,12.00
4,Finance,207,6.0,48.323671,2.00,27.50,0.000251,0.000000e+00,25.50
8,Materials / Mining / Chemicals,73,6.0,14.780822,3.00,24.00,0.000250,5.022576e-07,21.00
9,Media / Entertainment,66,6.0,26.151515,3.00,16.75,0.000265,0.000000e+00,13.75
16,Utilities,34,5.0,8.000000,1.00,11.50,0.000324,0.000000e+00,10.50
3,Energy,117,5.0,12.008547,2.00,12.00,0.000228,0.000000e+00,10.00
13,Technology / Telecom,301,5.0,30.916944,2.00,13.00,0.000227,0.000000e+00,11.00
7,Industrials / Aerospace / Defense,94,4.5,19.393617,2.00,13.75,0.000234,0.000000e+00,11.75


# Which companies are unusually central?

In [98]:
def iqr_outlier_table(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    threshold = q3 + 1.5 * iqr

    outliers = df[df[column] > threshold].copy()
    outliers = outliers.sort_values(column, ascending=False)

    return outliers, threshold

weighted_degree_outliers, wd_outlier_threshold = iqr_outlier_table(
    centrality_df,
    "weighted_degree"
)

print("Weighted degree outlier threshold:", wd_outlier_threshold)

weighted_degree_outliers[
    ["company", "industry_group", "degree", "weighted_degree", "pagerank", "betweenness"]
].head(30)


Weighted degree outlier threshold: 27.0


,company,industry_group,degree,weighted_degree,pagerank,betweenness
24,"Nasdaq, Inc.",Finance,633,1508,0.043951,0.537691
0,Apple Inc.,Technology / Telecom,246,1175,0.019776,0.258159
2,Amazon,Consumer / Retail / Food,207,1019,0.016822,0.256763
1,Google,Technology / Telecom,179,928,0.014813,0.146827
4,Microsoft,Technology / Telecom,196,838,0.014400,0.098145
6,Morgan Stanley,Finance,225,817,0.013932,0.179186
5,Goldman Sachs,Finance,199,707,0.011856,0.068028
3,Alphabet Inc.,Technology / Telecom,103,614,0.009445,0.082940
7,Citigroup,Finance,148,576,0.009514,0.058280
10,Deutsche Bank,Finance,162,570,0.009589,0.043150


In [102]:
weighted_degree_outliers["industry_group"].value_counts()

industry_group
Finance                              52
Technology / Telecom                 45
Consumer / Retail / Food             37
Healthcare                           26
Unknown                              22
Automotive                           14
Industrials / Aerospace / Defense    14
Media / Entertainment                14
Energy                               14
Materials / Mining / Chemicals       14
Transport / Logistics                 5
Other / Unmapped                      3
Travel / Hospitality                  3
Holding / Conglomerate                2
Real Estate                           2
Other / Unclear                       1
Agriculture                           1
Utilities                             1
Professional Services                 1
Name: count, dtype: int64

# Are popular companies also bridge companies

In [103]:
from scipy.stats import pearsonr
temp = centrality_df[["weighted_degree", "betweenness"]].dropna()


pearson_corr, pearson_p = pearsonr(
    temp["weighted_degree"],
    temp["betweenness"]
)


print("Pearson correlation:", pearson_corr)
print("Pearson p-value:", pearson_p)

Pearson correlation: 0.8617734592873533
Pearson p-value: 0.0


In [104]:
top_n = 20

top_weighted = centrality_df.sort_values(
    "weighted_degree",
    ascending=False
).head(top_n)

top_betweenness = centrality_df.sort_values(
    "betweenness",
    ascending=False
).head(top_n)

top_weighted_set = set(top_weighted["company_id"])
top_betweenness_set = set(top_betweenness["company_id"])

overlap = top_weighted_set & top_betweenness_set

print("Top weighted-degree companies:", len(top_weighted_set))
print("Top betweenness companies:", len(top_betweenness_set))
print("Overlap:", len(overlap))
print("Overlap share:", len(overlap) / top_n)

Top weighted-degree companies: 20
Top betweenness companies: 20
Overlap: 15
Overlap share: 0.75


In [105]:
centrality_df[
    centrality_df["company_id"].isin(overlap)
][
    ["company", "industry_group", "weighted_degree", "betweenness", "pagerank"]
].sort_values("betweenness", ascending=False)

,company,industry_group,weighted_degree,betweenness,pagerank
24,"Nasdaq, Inc.",Finance,1508,0.537691,0.043951
0,Apple Inc.,Technology / Telecom,1175,0.258159,0.019776
2,Amazon,Consumer / Retail / Food,1019,0.256763,0.016822
6,Morgan Stanley,Finance,817,0.179186,0.013932
1,Google,Technology / Telecom,928,0.146827,0.014813
4,Microsoft,Technology / Telecom,838,0.098145,0.014400
3,Alphabet Inc.,Technology / Telecom,614,0.082940,0.009445
26,Boeing,Industrials / Aerospace / Defense,390,0.076945,0.007516
15,AT&T,Technology / Telecom,433,0.068925,0.007438
5,Goldman Sachs,Finance,707,0.068028,0.011856


In [106]:
popular_not_bridge = top_weighted[
    ~top_weighted["company_id"].isin(top_betweenness_set)
]

popular_not_bridge[
    ["company", "industry_group", "weighted_degree", "betweenness", "pagerank"]
]

,company,industry_group,weighted_degree,betweenness,pagerank
16,Intel,Technology / Telecom,396,0.011036,0.006585
11,UBS,Finance,376,0.014972,0.006203
33,Tesla,Automotive,324,0.020026,0.005372
29,HSBC,Finance,323,0.013760,0.005218
28,Bank of America,Finance,311,0.011771,0.005169


In [48]:
bridge_not_popular = top_betweenness[
    ~top_betweenness["company_id"].isin(top_weighted_set)
]

bridge_not_popular[
    ["company", "industry_group", "weighted_degree", "betweenness", "pagerank"]
]

,company,industry_group,weighted_degree,betweenness,pagerank
73,Unilever,Consumer / Retail / Food,272,0.066862,0.004701
93,International,Automotive,120,0.042136,0.004218
34,Shell,Energy,136,0.036298,0.002757
40,Target Corporation,Consumer / Retail / Food,216,0.028692,0.004014
126,The Cooper Companies,Technology / Telecom,176,0.027959,0.004845


# Are centrality outliers concentrated in certain industries?

In [107]:
known_industry_centrality_df = centrality_df[
    ~centrality_df["industry_group"].isin(["Unknown", "Other / Unmapped"])
].copy()

known_industry_outliers = weighted_degree_outliers[
    ~weighted_degree_outliers["industry_group"].isin(["Unknown", "Other / Unmapped"])
].copy()

outlier_industry_counts = known_industry_outliers["industry_group"].value_counts()
all_industry_counts = known_industry_centrality_df["industry_group"].value_counts()

outlier_industry_comparison = pd.DataFrame({
    "all_companies": all_industry_counts,
    "centrality_outliers": outlier_industry_counts
}).fillna(0)

outlier_industry_comparison["all_company_share"] = (
    outlier_industry_comparison["all_companies"]
    / outlier_industry_comparison["all_companies"].sum()
)

outlier_industry_comparison["outlier_share"] = (
    outlier_industry_comparison["centrality_outliers"]
    / outlier_industry_comparison["centrality_outliers"].sum()
)

outlier_industry_comparison["overrepresentation"] = (
    outlier_industry_comparison["outlier_share"]
    / outlier_industry_comparison["all_company_share"]
)

outlier_industry_comparison = outlier_industry_comparison.sort_values(
    "centrality_outliers",
    ascending=False
)

outlier_industry_comparison

,all_companies,centrality_outliers,all_company_share,outlier_share,overrepresentation
industry_group,,,,,
Finance,207,52,0.147017,0.211382,1.437807
Technology / Telecom,280,45,0.198864,0.182927,0.919861
Consumer / Retail / Food,184,37,0.130682,0.150407,1.150937
Healthcare,153,26,0.108665,0.105691,0.972634
Media / Entertainment,66,14,0.046875,0.056911,1.214092
Energy,117,14,0.083097,0.056911,0.684872
Industrials / Aerospace / Defense,94,14,0.066761,0.056911,0.852448
Materials / Mining / Chemicals,73,14,0.051847,0.056911,1.097672
Automotive,49,14,0.034801,0.056911,1.635308


In [111]:
from scipy.stats import kruskal

metric = "weighted_degree"

groups = [
    group[metric].dropna().values
    for _, group in known_industry_centrality_df.groupby("industry_group")
    if len(group) >= 5
]

stat, p = kruskal(*groups)

print("Metric:", metric)
print("Kruskal-Wallis statistic:", stat)
print("p-value:", p)

Metric: weighted_degree
Kruskal-Wallis statistic: 34.52138731665993
p-value: 0.004618598945941724
